# Hallucination detection with Ω-SR (Omega Selective Release)

This tutorial implements **Ω-SR (Omega Selective Release)**, which turns a claim/evidence pair into one of two actions:

```text
release   the claim is safe to show
abstain   the system withholds the claim
```

The scorer is a real local verifier, `qwen2.5-coder:7b-instruct` through
Ollama. The threshold is selected on a small labelled calibration set, not on
the examples.

## Requirements

```bash
ollama pull qwen2.5-coder:7b-instruct
pip install -e '.[examples]'
```

The contract examples are public-procurement-style clauses, not legal advice.
The arXiv example uses the public abstract of *Attention Is All You Need*
(arXiv:1706.03762). The open-text examples are public historical summaries.

## The decision rule

The algorithm is **Ω-SR (Omega Selective Release)**: a calibrated truth
posterior plus a selective-risk threshold.

For claim `c` and evidence `E`:

```text
p(E) = P(claim supported | E)
release  iff  p(E) >= tau
abstain  otherwise
```

`alpha` is the maximum selective risk: the fraction of unsupported claims among
releases. `delta` is the confidence used by the cluster bootstrap. `tau` is the
calibrated threshold chosen on calibration data.

`alpha=0.10` means: on the calibration cohort, the release set should have at
most 10% unsupported claims, with cluster-level uncertainty accounted for.

## The four evidence layers (L0–L3)

The release score is only one part of the picture. Evidence can fail at four
different levels, and the level tells you what kind of warrant is missing.

| Layer | Question | Examples |
|---|---|---|
| **L0 — lexical alignment** | Do the words line up? | `thirty (30) days` vs `thirty days`; `may` vs `may not`; wrong unit, quantity, or offset |
| **L1 — semantic entailment** | Does the evidence entail the proposition? | `The Transformer uses recurrence` contradicts `based solely on attention mechanisms` |
| **L2 — epistemic settledness** | Is the evidence complete, current, consistent, and in scope? | A termination clause that omits the required notice period; an outdated policy; two sources that disagree |
| **L3 — domain/legal warrant** | Is the claim legally or professionally warranted? | What `for convenience` means in this jurisdiction; which clause prevails when documents conflict |

The layers are not confidence scores. They are kinds of warrant.

- **L0** catches surface mismatches: names, dates, units, negation, and roles.
- **L1** catches meaning-level failures: the claim is not entailed, or is
  defeated by the evidence.
- **L2** catches failures of the evidence itself: it may be incomplete,
  stale, contradictory, or outside its scope.
- **L3** catches failures that need domain authority: interpretation,
  jurisdiction, precedence, and professional judgement.

An important boundary: **L0–L2 can be assessed by a verifier against the
supplied evidence. L3 cannot be self-certified.** The party proposing a legal
or domain claim must not also be the disposer of that claim. L3 needs an
independent authority or process.

In this tutorial the LLM verifier approximates L0 and L1. It does **not**
certify L2 completeness or L3 warrant. That is why the deployment checklist
requires L3 provenance and an independent truth oracle. The contract examples
are illustrative clause-matching exercises, not legal advice.

In [1]:
import json
import os
import sys
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "mjolnir_ml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from sklearn.isotonic import IsotonicRegression
from mjolnir_ml import RiskControlledReleaseGate

MODEL = os.environ.get("MJOLNIR_SCORER_MODEL", "qwen2.5-coder:7b-instruct")
print("scorer:", MODEL)

scorer: qwen2.5-coder:7b-instruct


## Step 1 — ask a real verifier

The verifier receives the claim and the evidence. It must not use outside
knowledge. It returns two scores:

- `support`: evidence entails the claim;
- `contradiction`: evidence conflicts with the claim.

The raw score used below is:

```text
support × (1 − contradiction)
```

This is a real model judgement, not a lexical overlap heuristic. It is still
uncalibrated; calibration is the next step.

In [2]:
CACHE = {}

def ollama_json(prompt):
    body = json.dumps({
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "think": False,
        "options": {"temperature": 0, "seed": 0, "num_predict": 64},
    }).encode()
    request = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=body,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        text = json.loads(response.read()).get("response", "")
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1]) if 0 <= start < end else {}

In [3]:
def raw_support(claim, evidence):
    key = (claim, evidence)
    if key in CACHE:
        return CACHE[key]

    prompt = (
        'Return JSON {"support":0..1,"contradiction":0..1}. '
        "Judge the claim only against the evidence. support=1 means the "
        "evidence entails the claim; contradiction=1 means it conflicts; "
        "0.5/0.5 means uncertain. Do not use outside knowledge. "
        f"Claim: {claim}\nEvidence: {evidence}"
    )
    payload = ollama_json(prompt)
    support = max(0.0, min(1.0, float(payload.get("support", 0.0))))
    contradiction = max(0.0, min(1.0,
                                float(payload.get("contradiction", 0.0))))
    score = max(0.0, min(1.0, support * (1.0 - contradiction)))
    CACHE[key] = score
    return score

## Step 2 — calibrate the verifier

A model that says `0.8` does not necessarily mean an 80% chance of support.
We fit isotonic regression on labelled calibration rows.

Calibration rows are separate from the five examples. Each row records:

```text
(claim, evidence, outcome, cluster)
outcome = 1  supported
outcome = 0  not supported
cluster = shared-evidence unit
```

In [4]:
CALIBRATION = [
    ("Delivery must occur within thirty days.",
     "Delivery must occur within thirty days.", 1, "contract-a"),
    ("Payment is due within ten days.",
     "Payment is due within ten days.", 1, "contract-b"),
    ("The Transformer is based solely on attention mechanisms.",
     "The Transformer is based solely on attention mechanisms.", 1, "arxiv"),
    ("Paris is the capital of France.",
     "Paris is the capital of France.", 1, "open-a"),
    ("The Transformer uses recurrence.",
     "Attention is sufficient.", 0, "arxiv"),
    ("The moon is made of cheese.",
     "Lunar regolith is rocky.", 0, "open-b"),
    ("The supplier may terminate immediately.",
     "Notice is required.", 0, "contract-a"),
    ("Anne Frank died in March 1945.",
     "The date was earlier.", 0, "open-b"),
]

In [5]:
cal_raw = [raw_support(claim, evidence)
           for claim, evidence, _, _ in CALIBRATION]
cal_outcomes = [outcome for _, _, outcome, _ in CALIBRATION]
cal_clusters = [cluster for _, _, _, cluster in CALIBRATION]

calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(cal_raw, cal_outcomes)
cal_p = calibrator.predict(cal_raw)

print(f"{'raw':>6} {'calibrated':>10} outcome")
for raw, p, outcome in zip(cal_raw, cal_p, cal_outcomes):
    print(f"{raw:6.3f} {p:10.3f} {outcome:>7}")

   raw calibrated outcome
 1.000      1.000       1
 1.000      1.000       1
 1.000      1.000       1
 1.000      1.000       1
 0.000      0.000       0
 0.000      0.000       0
 0.250      0.000       0
 0.250      0.000       0


## Step 3 — fit the risk-controlled release gate

`RiskControlledReleaseGate` is the independent implementation of **Ω-SR
(Omega Selective Release)**.

The name says what it does:

- **risk-controlled**: `alpha` is the maximum selective risk — the fraction of
  unsupported claims among releases;
- **release gate**: the output is binary, `release` or `abstain`;
- **calibrated**: it consumes `P(supported | claim, evidence)`, not raw model
  confidence;
- **independent**: it is a standalone library component, not part of
  `DecisionModel`.

It learns `tau` by maximizing coverage subject to the selective-risk bound.

Parameters:

- `alpha`: maximum selective risk;
- `delta`: bootstrap confidence level;
- `clusters`: optional; resample evidence clusters, not rows;
- `tau`: the learned probability threshold.

If no threshold meets `alpha`, `tau` is `None` and every decision abstains. That
is fail-closed behavior, not a tuning failure.

What the gate does **not** do:

- fit or call a language model;
- determine truth;
- prove that the evidence entails the claim;
- supply L3 legal or domain authority;
- enforce independence of the scorer or calibration labels.

Those remain the caller's responsibility.

In [6]:
rel = RiskControlledReleaseGate(alpha=0.10, delta=0.05, seed=0)
rel.fit(cal_p, cal_outcomes, clusters=cal_clusters)
print(rel.certificate.as_dict())

{'schema': 'mjolnir.ml.abstention/1', 'alpha': 0.1, 'delta': 0.05, 'tau': 1.0, 'coverage': 0.5, 'risk': 0.0, 'upper': 0.0, 'n': 8, 'n_clusters': 5, 'mode': 'cluster'}


## Step 4 — five held-out examples

The examples cover:

- two public-contract clauses;
- one RAG claim over an arXiv abstract;
- two open-text claims.

Expected behavior:

- supported examples should `release`;
- contradicted examples should `abstain`.

The examples are not used to fit the threshold.

In [7]:
EXAMPLES = [
    ("public contract",
     "The supplier shall deliver the goods within thirty (30) days.",
     "The supplier shall deliver the goods within thirty (30) days of the "
     "purchase order.",
     "release"),
    ("public contract",
     "The supplier may terminate immediately without notice.",
     "Either party may terminate for convenience with thirty (30) days "
     "prior written notice.",
     "abstain"),
    ("arXiv RAG",
     "The Transformer uses recurrence and convolution.",
     "We propose a new simple network architecture, the Transformer, based "
     "solely on attention mechanisms, dispensing with recurrence and "
     "convolutions entirely. arXiv:1706.03762.",
     "abstain"),
    ("open text",
     "Anne Frank died of typhus in Bergen-Belsen.",
     "Anne Frank died of typhus in the Bergen-Belsen concentration camp.",
     "release"),
    ("open text",
     "Anne Frank died in March 1945.",
     "Researchers concluded that Anne and Margot probably did not survive "
     "to March 1945.",
     "abstain"),
]

## Step 5 — run and compare

For each example we:

1. score the claim against its evidence;
2. map the raw score through the calibrator;
3. ask `RiskControlledReleaseGate` for a decision;
4. compare the result with the expected outcome.

The assertion makes the tutorial fail loudly if the calibrated policy and the
examples disagree.

In [8]:
print(f"{'kind':<18} {'p':>5} {'verdict':<8} {'expected':<9} claim")
print("-" * 88)

for kind, claim, evidence, expected in EXAMPLES:
    raw = raw_support(claim, evidence)
    p = float(calibrator.predict([raw])[0])
    decision = rel.decide(p, cluster=kind, claim=claim)
    print(f"{kind:<18} {p:5.3f} {decision.verdict:<8} "
          f"{expected:<9} {claim}")
    assert decision.verdict == expected, (claim, decision.verdict, expected)

kind                   p verdict  expected  claim
----------------------------------------------------------------------------------------


public contract    1.000 release  release   The supplier shall deliver the goods within thirty (30) days.


public contract    0.000 abstain  abstain   The supplier may terminate immediately without notice.


arXiv RAG          0.000 abstain  abstain   The Transformer uses recurrence and convolution.


open text          1.000 release  release   Anne Frank died of typhus in Bergen-Belsen.


open text          0.000 abstain  abstain   Anne Frank died in March 1945.


## Reading the result

The certificate reports:

```text
alpha   requested maximum selective risk
tau     calibrated release threshold
coverage fraction of calibration claims released
risk    unsupported fraction among releases
mode    cluster-aware or iid
```

In the expected run, supported examples release and contradicted examples
abstain.

Limits of this tutorial:

- eight calibration rows are enough to demonstrate the mechanics, not to deploy;
- the LLM judge is not a domain authority;
- the contract text is illustrative, not legal advice;
- a production system needs a larger held-out calibration set and an
  independent audit split.

Replace `raw_support` with a production NLI/LLM verifier exposing the same
`P(supported | claim, evidence)` interface. The release policy stays the same.

## What would make this deployable?

This notebook is a correctness demo, not a deployment. A deployable release
policy needs:

1. **A real calibration cohort.** Hundreds or thousands of labelled claims, not
   eight.
2. **Source-held-out calibration and audit.** No source or response may appear
   in both splits.
3. **An independent truth oracle.** Human adjudication or a validated domain
   process, not the verifier being certified.
4. **Cluster-level risk accounting.** Evidence is shared; row-level counts
   understate uncertainty.
5. **A frozen policy version.** Model, prompt, calibrator, threshold, alpha and
   cluster definition all travel with the certificate.
6. **Fail-closed behavior.** Missing calibration, no threshold, drift, or a
   failed health check must abstain.
7. **Monitoring.** Track selective risk, coverage, abstention, and cluster-level
   failures after deployment.
8. **Drift and rollback.** Recalibrate when the evidence or model distribution
   shifts; roll back if audit risk exceeds the bound.
9. **L3 provenance** for claims that require domain or legal authority.
10. **Tests and sign-off.** Unit tests, replayable certificates, and a named
    owner for the risk budget.

The sample below wraps the fitted policy in a minimal deployment gate. It is
deliberately small; the checklist above is the real deployment work.

In [9]:
from mjolnir_ml import ABSTAIN, ReleaseDecision

def readiness(rel, *, min_calibration=100, min_clusters=50,
              max_alpha=0.10):
    cert = rel.certificate
    checks = {
        "has_threshold": cert.tau is not None,
        "calibration_size": cert.n >= min_calibration,
        "cluster_size": (cert.n_clusters or 0) >= min_clusters,
        "alpha_within_budget": cert.alpha <= max_alpha,
        "cluster_mode": cert.mode == "cluster",
    }
    return all(checks.values()), checks

In [10]:
class ReleasePipeline:
    """Minimal deployment wrapper: frozen policy + fail-closed decisions."""

    def __init__(self, scorer, calibrator, release, *, model_id,
                 policy_version, min_calibration=100, min_clusters=50):
        self.scorer = scorer
        self.calibrator = calibrator
        self.release = release
        self.model_id = model_id
        self.policy_version = policy_version
        self.min_calibration = min_calibration
        self.min_clusters = min_clusters

    def ready(self):
        return readiness(self.release,
                        min_calibration=self.min_calibration,
                        min_clusters=self.min_clusters)

    def decide(self, claim, evidence, cluster):
        ok, checks = self.ready()
        if not ok:
            return ReleaseDecision(ABSTAIN, 0.0, self.release.tau,
                                   cluster=cluster, claim=claim)
        raw = self.scorer(claim, evidence)
        p = float(self.calibrator.predict([raw])[0])
        return self.release.decide(p, cluster=cluster, claim=claim)

In [11]:
pipeline = ReleasePipeline(
    scorer=raw_support,
    calibrator=calibrator,
    release=rel,
    model_id=MODEL,
    policy_version="omega-sr-hallucination-tutorial/1",
    min_calibration=8,
    min_clusters=5,
)
ok, checks = pipeline.ready()
print("ready:", ok)
print("checks:", checks)

for kind, claim, evidence, expected in EXAMPLES[:2]:
    d = pipeline.decide(claim, evidence, cluster=kind)
    print(f"{kind:<18} {d.probability:5.3f} {d.verdict:<8} {claim}")

ready: True
checks: {'has_threshold': True, 'calibration_size': True, 'cluster_size': True, 'alpha_within_budget': True, 'cluster_mode': True}
public contract    1.000 release  The supplier shall deliver the goods within thirty (30) days.
public contract    0.000 abstain  The supplier may terminate immediately without notice.


The pipeline gate passes only because the demo lowers `min_calibration` to 8
and `min_clusters` to 5. In production those numbers must come from the risk
budget and audit design, not from the tutorial.

The fail-closed branch is the important part: if readiness fails, `decide`
returns `abstain` without calling the scorer or the policy.

## Novelty claim

The novelty of Ω-SR is **architectural, not statistical**.

The threshold rule is deliberately conventional: calibrate a truth posterior,
then threshold it under a selective-risk budget. In iid mode that is conformal
risk control; in cluster mode it is a cluster-bootstrap upper bound. Ω-SR does
not claim a new finite-sample risk estimator.

In one sentence: **Ω-SR is a release pinned against a policy that is
layer-certified by an independent oracle, with L3 claims barred from
self-certification.**

The independent oracle is necessary but **not** the novelty. CRC, LTT,
selective classification, and conformal abstention already calibrate against
held-out labels supplied by an external correctness oracle. The differentiating
claim is that the release policy is gated by the L0–L3 warrant hierarchy, and
that domain/legal warrant (L3) cannot be issued by the same party that proposes
the claim.

The contribution is the **release contract**:

1. an external truth oracle, never the verifier itself;
2. a calibrated `P(supported | claim, evidence)` posterior;
3. cluster-aware threshold selection for shared evidence;
4. a binary `release` / `abstain` action;
5. a portable certificate containing `alpha`, `delta`, `tau`, coverage,
   selective risk, upper bound, and cluster count;
6. a fail-closed readiness gate;
7. an L0–L3 evidence hierarchy in which L3 claims cannot be self-certified.

To our knowledge, prior work supplies these pieces separately, but does not
package them into one deployable refusal contract with an explicit
proposer-versus-disposer separation. That composition, and the L3 refusal
boundary, is the novelty claim to defend. It is not a claim that thresholding
itself is new.

## Further reading

- Chow, C. K. (1970). *On optimum recognition error and reject tradeoff.*
  IEEE Transactions on Information Theory. The original reject option.
- El-Yaniv, R., & Wiener, Y. (2010). *On the foundations of noise-free
  selective classification.* JMLR. Foundations of selective prediction.
- Geifman, Y., & El-Yaniv, R. (2017). *Selective Classification for Deep Neural
  Networks.* NeurIPS. The modern confidence-threshold baseline.
- Angelopoulos, A. N., et al. (2022). *Conformal Risk Control.*
  arXiv:2208.02814. The finite-sample risk-control threshold used in iid mode.
- Angelopoulos, A. N., et al. (2021). *Learn then Test: Calibrating Predictive
  Algorithms to Achieve Risk Control.* arXiv:2110.01052. Alternative
  multiple-testing certification.
- Fisch, A., Jaakkola, T., & Barzilay, R. (2022). *Calibrated Selective
  Classification.* arXiv:2208.12084. Calibration-aware selective prediction.
- *Mitigating LLM Hallucinations via Conformal Abstention.* arXiv:2405.01563.
  Conformal abstention with a hallucination-risk bound.
- *UniCR: Trusted Uncertainty in Large Language Models.* arXiv:2509.01455.
  Calibrated correctness probability plus risk-controlled refusal; the closest
  LLM pipeline to Ω-SR.
- *Selective Conformal Risk Control.* arXiv:2512.12844. Direct naming and
  framing neighbor.
- *Conformal Selective Prediction with General Risk Control (SCoRE).*
  arXiv:2603.24704. E-value-based selective risk control.
- *Fixed-Sequence Confidence-Interval Calibration for Finite-Sample Selective
  Risk Control.* DOI:10.3390/stats9050109. Confidence-bound threshold
  certification for LLM selective answering.
- Tayebati, M., et al. *CAP: Conformalized Abstention Policies.* PMLR v304.
  Adaptive, utility-driven abstention policies.